# Wasabi Sync — Download Notebook

| Cell | Purpose |
|------|---------|
| **1** | Install & Imports |
| **2** | Configuration (credentials + region) |
| **3** | All helper functions (run once) |
| **4** | ▶ Download a single file |
| **5** | ▶ Download a folder |
| **6** | ▶ Batch download from a key list |
| **A** | Appendix — Browse bucket contents |

## 1 — Install & Imports

In [ ]:
%pip install boto3 tqdm ipywidgets python-dotenv --quiet

In [1]:
import os
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Optional

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from tqdm import tqdm  # works everywhere; no ipywidgets needed

try:
    from dotenv import load_dotenv
    _dotenv_path = Path("..") / ".env"
    load_dotenv(_dotenv_path, override=True)
    print(f"✓ .env loaded from {_dotenv_path.resolve()}")
except Exception as _e:
    print(f"  .env not loaded: {_e}")

✓ .env loaded from /home/taiaburrahman/dataset_manager_pro/.env


## 2 — Configuration

Values are read from the `.env` file automatically.  
Override any value here by assigning directly (overrides the env file).

In [2]:
# ── Wasabi credentials ─────────────────────────────────────────────────────
# Leave as os.getenv(...) to read from .env, or paste values directly.
WASABI_ACCESS_KEY_ID     = os.getenv("WASABI_ACCESS_KEY_ID",     "")
WASABI_SECRET_ACCESS_KEY = os.getenv("WASABI_SECRET_ACCESS_KEY", "")
WASABI_BUCKET            = os.getenv("WASABI_BUCKET",            "")
WASABI_REGION            = os.getenv("WASABI_REGION",            "us-east-1")
WASABI_ENDPOINT          = os.getenv("WASABI_ENDPOINT",          "https://s3.wasabisys.com")

# ── Parallel download threads ──────────────────────────────────────────────
MAX_WORKERS = 4

print(f"Bucket  : {WASABI_BUCKET}")
print(f"Region  : {WASABI_REGION}")
print(f"Endpoint: {WASABI_ENDPOINT}")

Bucket  : bbvision
Region  : us-east-2
Endpoint: https://s3.us-east-2.wasabisys.com


### Auto-detect Region  
Run this cell **only if you see a 400 Bad Request** error.  
It probes every Wasabi regional endpoint and patches `WASABI_REGION` / `WASABI_ENDPOINT` in the session.

In [ ]:
_ALL_ENDPOINTS = [
    ("us-east-1",      "https://s3.us-east-1.wasabisys.com"),
    ("us-east-2",      "https://s3.us-east-2.wasabisys.com"),
    ("us-west-1",      "https://s3.us-west-1.wasabisys.com"),
    ("ca-central-1",   "https://s3.ca-central-1.wasabisys.com"),
    ("eu-west-1",      "https://s3.eu-west-1.wasabisys.com"),
    ("eu-central-1",   "https://s3.eu-central-1.wasabisys.com"),
    ("eu-west-2",      "https://s3.eu-west-2.wasabisys.com"),
    ("eu-south-1",     "https://s3.eu-south-1.wasabisys.com"),
    ("ap-northeast-1", "https://s3.ap-northeast-1.wasabisys.com"),
    ("ap-northeast-2", "https://s3.ap-northeast-2.wasabisys.com"),
    ("ap-southeast-1", "https://s3.ap-southeast-1.wasabisys.com"),
    ("ap-southeast-2", "https://s3.ap-southeast-2.wasabisys.com"),
]

_cfg = Config(signature_version="s3v4", retries={"max_attempts": 1, "mode": "standard"})
print(f"Probing bucket '{WASABI_BUCKET}' across all Wasabi regions …\n")

_found = False
for _region, _endpoint in _ALL_ENDPOINTS:
    _c = boto3.client("s3", endpoint_url=_endpoint, region_name=_region,
                      aws_access_key_id=WASABI_ACCESS_KEY_ID,
                      aws_secret_access_key=WASABI_SECRET_ACCESS_KEY, config=_cfg)
    try:
        _c.head_bucket(Bucket=WASABI_BUCKET)
        _found = True
    except ClientError as _e:
        if _e.response["Error"]["Code"] == "403":
            _found = True  # correct region, just no HeadBucket permission
    if _found:
        WASABI_REGION   = _region
        WASABI_ENDPOINT = _endpoint
        os.environ["WASABI_REGION"]   = _region
        os.environ["WASABI_ENDPOINT"] = _endpoint
        print(f"✓ Found!  region={_region}")
        print(f"          endpoint={_endpoint}")
        print(f"\nAdd to your .env:")
        print(f"  WASABI_REGION={_region}")
        print(f"  WASABI_ENDPOINT={_endpoint}")
        break

if not _found:
    print("✗ Bucket not found. Check credentials and bucket name.")

## 3 — Helper Functions

Run this cell once. It defines all internals used by the download cells below.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# ALL HELPER FUNCTIONS — run once before any download cell
# ══════════════════════════════════════════════════════════════════════════════

def _g(name: str, default: str = "") -> str:
    """Read a config value from module globals → os.environ → default."""
    return globals().get(name) or os.getenv(name, default)

def _bucket() -> str:
    return _g("WASABI_BUCKET", "")

def _workers() -> int:
    return globals().get("MAX_WORKERS", 4)

def get_client() -> boto3.client:
    """Build a boto3 S3 client pointed at the configured Wasabi endpoint."""
    return boto3.client(
        "s3",
        endpoint_url=_g("WASABI_ENDPOINT", "https://s3.wasabisys.com"),
        region_name=_g("WASABI_REGION", "us-east-1"),
        aws_access_key_id=_g("WASABI_ACCESS_KEY_ID"),
        aws_secret_access_key=_g("WASABI_SECRET_ACCESS_KEY"),
        config=Config(signature_version="s3v4",
                      retries={"max_attempts": 3, "mode": "adaptive"}),
    )

def _human_size(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"

def _box(lines: list[str], width: int = 68) -> str:
    """Render a bordered summary box."""
    sep   = "─" * (width - 2)
    top   = f"╔{sep}╗"
    bot   = f"╚{sep}╝"
    mid   = f"╠{sep}╣"
    def row(s):
        return f"║  {s:<{width - 4}}║"
    out = [top]
    for line in lines:
        if line == "---":
            out.append(mid)
        else:
            out.append(row(line))
    out.append(bot)
    return "\n".join(out)

def test_connection() -> bool:
    """Verify credentials and bucket access."""
    client = get_client()
    bucket = _bucket()
    try:
        client.head_bucket(Bucket=bucket)
        print(f"✓ Connected — bucket '{bucket}' is accessible")
        return True
    except ClientError as e:
        code = e.response["Error"]["Code"]
        print(f"✗ Connection failed ({code}): {e}")
        return False

def list_bucket(prefix: str = "") -> list[dict]:
    """Return all objects under *prefix* as a list of dicts."""
    client = get_client()
    objects = []
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=_bucket(), Prefix=prefix):
        for obj in page.get("Contents", []):
            objects.append({
                "key":           obj["Key"],
                "size":          obj["Size"],
                "last_modified": obj["LastModified"].isoformat(),
                "etag":          obj["ETag"].strip('"'),
            })
    return objects

def download_file(
    source_key: str,
    save_path: str | Path,
    overwrite: bool = False,
) -> Path:
    """
    Download a single file from the bucket.

    Parameters
    ----------
    source_key : Full S3 object key, e.g. "/Image_Features_DBs/file.parquet"
    save_path  : Local destination path, e.g. "/data/file.parquet"
    overwrite  : Re-download even if the file already exists.
    """
    client = get_client()
    dest   = Path(save_path)
    key    = source_key.lstrip("/")

    if dest.exists() and not overwrite:
        size_str = _human_size(dest.stat().st_size)
        print(_box([
            "DOWNLOAD SKIPPED",
            "---",
            f"File     : {Path(key).name}",
            f"Size     : {size_str}",
            f"Saved at : {dest}",
            "---",
            "⏭  Already exists — set OVERWRITE = True to re-download",
        ]))
        return dest

    try:
        meta        = client.head_object(Bucket=_bucket(), Key=key)
        total_bytes = meta["ContentLength"]
    except ClientError:
        total_bytes = None

    dest.parent.mkdir(parents=True, exist_ok=True)

    with tqdm(total=total_bytes, unit="B", unit_scale=True,
              unit_divisor=1024, desc=Path(key).name) as bar:
        client.download_file(_bucket(), key, str(dest),
                             Callback=lambda n: bar.update(n))

    size_str = _human_size(total_bytes) if total_bytes else _human_size(dest.stat().st_size)
    print(_box([
        "DOWNLOAD COMPLETE",
        "---",
        f"File     : {Path(key).name}",
        f"Size     : {size_str}",
        f"Saved at : {dest}",
    ]))
    return dest


# ── Folder download ────────────────────────────────────────────────────────────

@dataclass
class _FolderResult:
    prefix:       str
    save_path:    str  = ""
    total:        int  = 0
    total_size:   int  = 0
    succeeded:    int  = 0
    skipped_keys: list[str] = field(default_factory=list)
    failed:       list[str] = field(default_factory=list)

    @property
    def skipped(self) -> int:
        return len(self.skipped_keys)

    def print_summary(self):
        sub_folders = {k.rsplit("/", 1)[0] for k in self.skipped_keys + self.failed
                       if "/" in k}
        lines = [
            "FOLDER DOWNLOAD SUMMARY",
            "---",
            f"Source prefix  : {self.prefix}",
            f"Saved to       : {self.save_path}",
            "---",
            f"Total files    : {self.total}  ({_human_size(self.total_size)})",
            f"✓  Downloaded  : {self.succeeded}",
            f"⏭  Skipped     : {self.skipped}",
            f"✗  Failed      : {len(self.failed)}",
        ]
        if self.skipped_keys:
            lines += ["---", "Skipped files (already existed):"]
            for k in self.skipped_keys:
                lines.append(f"   · {Path(k).name}")
        if self.failed:
            lines += ["---", "Failed files:"]
            for f in self.failed:
                lines.append(f"   · {f}")
        print(_box(lines))


def download_folder(
    source_prefix: str,
    save_path: str | Path,
    overwrite: bool = False,
    strip_prefix: bool = True,
) -> _FolderResult:
    """
    Download every object that starts with *source_prefix*.

    Parameters
    ----------
    source_prefix : S3 prefix (virtual folder), e.g. "/datasets/train/"
    save_path     : Local directory to save into, e.g. "/data/train/"
    overwrite     : Re-download files that already exist.
    strip_prefix  : True  → save_path/filename.ext
                    False → save_path/full/s3/prefix/filename.ext
    """
    client = get_client()
    prefix = source_prefix.lstrip("/")
    if prefix and not prefix.endswith("/"):
        prefix += "/"

    local_root = Path(save_path)

    print(f"Listing objects under prefix: '{prefix}' …")
    objects = list_bucket(prefix=prefix)

    if not objects:
        print("No objects found under that prefix.")
        return _FolderResult(prefix=prefix, save_path=str(local_root))

    # ── Content breakdown ──────────────────────────────────────────────────
    total_bytes  = sum(o["size"] for o in objects)
    sub_prefixes = {o["key"][len(prefix):].split("/")[0]
                    for o in objects if "/" in o["key"][len(prefix):]}
    n_files      = sum(1 for o in objects if "/" not in o["key"][len(prefix):])
    n_subfolders = len(sub_prefixes)

    print(f"{'─'*50}")
    print(f"  Files at root level : {n_files}")
    print(f"  Sub-folders         : {n_subfolders}"
          + (f"  {sorted(sub_prefixes)}" if sub_prefixes else ""))
    print(f"  Total objects       : {len(objects)}  ({_human_size(total_bytes)})")
    print(f"{'─'*50}\n")

    result = _FolderResult(
        prefix=prefix, save_path=str(local_root),
        total=len(objects), total_size=total_bytes,
    )
    lock = threading.Lock()

    overall = tqdm(total=total_bytes, unit="B", unit_scale=True,
                   unit_divisor=1024, desc="Overall", position=0)

    def _one(obj):
        key = obj["key"]
        rel = key[len(prefix):] if strip_prefix else key

        # Skip empty rel (prefix itself) or S3 folder-placeholder keys
        if not rel or rel.endswith("/") or obj["size"] == 0:
            return "skip", key

        dest = local_root / rel

        # Skip if the destination path is already a directory (folder placeholder
        # objects can collide with directories created by parallel workers)
        if dest.is_dir():
            overall.update(obj["size"])
            return "skip", key

        if dest.exists() and not overwrite:
            overall.update(obj["size"])
            return "skip", key

        try:
            dest.parent.mkdir(parents=True, exist_ok=True)
        except FileExistsError:
            # Another thread may have created it between the check and mkdir
            pass

        downloaded = [0]

        def _cb(n):
            downloaded[0] += n
            overall.update(n)

        try:
            client.download_file(_bucket(), key, str(dest), Callback=_cb)
            rem = obj["size"] - downloaded[0]
            if rem > 0:
                overall.update(rem)
            return "ok", key
        except Exception as exc:
            rem = obj["size"] - downloaded[0]
            if rem > 0:
                overall.update(rem)
            return "fail", f"{key} — {exc}"

    with ThreadPoolExecutor(max_workers=_workers()) as pool:
        futures = {pool.submit(_one, obj): obj for obj in objects}
        for future in tqdm(as_completed(futures), total=len(objects),
                           desc="Files", position=1, leave=True):
            status, info = future.result()
            with lock:
                if status == "ok":     result.succeeded += 1
                elif status == "skip": result.skipped_keys.append(info)
                else:                  result.failed.append(info)

    overall.close()
    print()
    result.print_summary()
    return result


# ── Batch download ─────────────────────────────────────────────────────────────

def batch_download(
    source_keys: list[str],
    save_path: str | Path,
    overwrite: bool = False,
) -> dict:
    """
    Download an explicit list of S3 keys in parallel.

    Parameters
    ----------
    source_keys : List of full S3 keys to download.
    save_path   : Local root directory; each key's path is mirrored under it.
    overwrite   : Re-download if the local file already exists.
    """
    client       = get_client()
    local_root   = Path(save_path)
    skipped_keys = []
    failed       = []
    succeeded    = 0
    lock         = threading.Lock()

    def _one(raw_key: str):
        key  = raw_key.lstrip("/")
        dest = local_root / key
        if dest.exists() and not overwrite:
            return "skip", key
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            client.download_file(_bucket(), key, str(dest))
            return "ok", key
        except Exception as exc:
            return "fail", f"{key} — {exc}"

    with ThreadPoolExecutor(max_workers=_workers()) as pool:
        futures = {pool.submit(_one, k): k for k in source_keys}
        for future in tqdm(as_completed(futures), total=len(source_keys), desc="Batch"):
            status, info = future.result()
            with lock:
                if status == "ok":     succeeded    += 1
                elif status == "skip": skipped_keys.append(info)
                else:                  failed.append(info)

    lines = [
        "BATCH DOWNLOAD SUMMARY",
        "---",
        f"Total files  : {len(source_keys)}",
        f"Saved to     : {local_root}",
        "---",
        f"✓  Downloaded : {succeeded}",
        f"⏭  Skipped    : {len(skipped_keys)}",
        f"✗  Failed     : {len(failed)}",
    ]
    if skipped_keys:
        lines += ["---", "Skipped files (already existed):"]
        for k in skipped_keys:
            lines.append(f"   · {Path(k).name}")
    if failed:
        lines += ["---", "Failed files:"]
        for f in failed:
            lines.append(f"   · {f}")
    print()
    print(_box(lines))
    return {"succeeded": succeeded, "skipped": skipped_keys, "failed": failed}


# ── verify connection on load ──────────────────────────────────────────────────
test_connection()

✓ Connected — bucket 'bbvision' is accessible


True

## 4 — Download a Single File

Set `SOURCE_KEY` to the full S3 path of the file and `SAVE_PATH` to where you want it saved locally.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_KEY = "/Image_Features_DBs/GAID_Image/GAID_Dataset_v9_Val_features.parquet"
SAVE_PATH  = "/home/taiaburrahman/dataset_manager_pro/data/GAID_Dataset_v9_Val_features.parquet"
OVERWRITE  = False   # set True to re-download even if the file exists
# ───────────────────────────────────────────────────────────────────────────

download_file(SOURCE_KEY, SAVE_PATH, overwrite=OVERWRITE)

## 5 — Download a Folder

Set `SOURCE_FOLDER` to the S3 prefix (virtual folder) and `SAVE_PATH` to the local directory.  
All files under the prefix will be downloaded in parallel.

In [4]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_FOLDER = "/datasets/GenAI_Image_Database/Generated_Data_2025/"   # S3 prefix / virtual folder
SAVE_PATH     = "/home/taiaburrahman/dataset_manager_pro/data/wasabi/GenAI_Image_Database/Generated_Data_2025/"       # local destination directory
OVERWRITE     = False   # set True to re-download existing files
STRIP_PREFIX  = True    # True  → SAVE_PATH/filename.ext
                        # False → SAVE_PATH/full/s3/prefix/filename.ext
# ───────────────────────────────────────────────────────────────────────────

download_folder(SOURCE_FOLDER, SAVE_PATH,
                overwrite=OVERWRITE, strip_prefix=STRIP_PREFIX)

Listing objects under prefix: 'datasets/GenAI_Image_Database/Generated_Data_2025/' …
──────────────────────────────────────────────────
  Files at root level : 17
  Sub-folders         : 0
  Total objects       : 17  (15.7 GB)
──────────────────────────────────────────────────



Files: 100%|██████████| 17/17 [02:13<00:00,  7.83s/it]7MB/s]   
Overall: 100%|██████████| 15.7G/15.7G [02:13<00:00, 126MB/s]


╔──────────────────────────────────────────────────────────────────╗
║  FOLDER DOWNLOAD SUMMARY                                         ║
╠──────────────────────────────────────────────────────────────────╣
║  Source prefix  : datasets/GenAI_Image_Database/Generated_Data_2025/║
║  Saved to       : /home/taiaburrahman/dataset_manager_pro/data/wasabi/GenAI_Image_Database/Generated_Data_2025║
╠──────────────────────────────────────────────────────────────────╣
║  Total files    : 17  (15.7 GB)                                  ║
║  ✓  Downloaded  : 16                                             ║
║  ⏭  Skipped     : 1                                              ║
║  ✗  Failed      : 0                                              ║
╠──────────────────────────────────────────────────────────────────╣
║  Skipped files (already existed):                                ║
║     · Generated_Data_2025                                        ║
╚─────────────────────────────────────────────────────

_FolderResult(prefix='datasets/GenAI_Image_Database/Generated_Data_2025/', save_path='/home/taiaburrahman/dataset_manager_pro/data/wasabi/GenAI_Image_Database/Generated_Data_2025', total=17, total_size=16813205366, succeeded=16, skipped_keys=['datasets/GenAI_Image_Database/Generated_Data_2025/'], failed=[])

## 6 — Batch Download

List any number of S3 keys to download in parallel. All files land under `SAVE_PATH`, mirroring the S3 key structure.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_KEYS = [
    "/Image_Features_DBs/GAID_Image/GAID_Dataset_v9_Val_features.parquet",
    "/Image_Features_DBs/GAID_Image/GAID_Dataset_v9_sub_Val_features.parquet",
    "/backup/A10_ext_HASNAT/video_infer_results.tar.gz",
]
https://s3.us-east-2.wasabisys.com/bbvision/datasets/gen_ai_detector_dataset_scaled_224/genAI_3.tar.lz4
SAVE_PATH = "/data/downloads/"   # local root; S3 key structure is mirrored under it
OVERWRITE = False
# ───────────────────────────────────────────────────────────────────────────

batch_download(SOURCE_KEYS, SAVE_PATH, overwrite=OVERWRITE)

## Appendix — Browse Bucket Contents

Lists objects in the bucket. Set `BROWSE_PREFIX` to narrow to a sub-folder.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
BROWSE_PREFIX = ""   # e.g. "/Image_Features_DBs/" to narrow the listing
# ───────────────────────────────────────────────────────────────────────────

objects = list_bucket(prefix=BROWSE_PREFIX.lstrip("/"))

if not objects:
    print("No objects found.")
else:
    total_size = sum(o["size"] for o in objects)
    print(f"{'Key':<70} {'Size':>10}  Last Modified")
    print("-" * 100)
    for o in objects:
        print(f"{o['key']:<70} {_human_size(o['size']):>10}  {o['last_modified'][:19]}")
    print("-" * 100)
    print(f"Total: {len(objects)} object(s)  —  {_human_size(total_size)}")